In [1]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.5
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 3. Transformación de bloques en matrices X/y con split temporal 70/30
# DESCRIPCIÓN:
#   Transforma la tabla de bloques de ocupación en matrices de features X
#   y etiquetas y, realizando un split temporal 70/30 sin solapamiento.
#   NOVEDAD: Incluye Minutes_Since_Start y *_trend como features
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

# Directorio
OUTPUT_DIR = 'ml_features_grouped'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===================================================================
# 1. CARGA DE TABLA DE BLOQUES DE OCUPACIÓN
# ===================================================================
print("\n" + "="*60)
print("1. LOADING OCCUPANCY BLOCKS TABLE")
print("="*60)

INPUT_FILE = 'occupancy_analysis/1.5_occupancy_blocks_analysis.xlsx'

if not os.path.exists(INPUT_FILE):
    print(f"ERROR: File not found: {INPUT_FILE}")
    exit()

df = pd.read_excel(INPUT_FILE)
df['Date'] = pd.to_datetime(df['Date'])

print(f"   Rows: {len(df)}")
print(f"   Columns: {len(df.columns)}")
print(f"   Occupancy range: {df['Occupancy'].min():.0f} - {df['Occupancy'].max():.0f}")
print(f"   Date range: {df['Date'].min().date()} - {df['Date'].max().date()}")

# ===================================================================
# 2. IDENTIFICACIÓN DE COLUMNAS DE SENSORES
# ===================================================================
print("\n" + "="*60)
print("2. IDENTIFYING FEATURE COLUMNS")
print("="*60)

# Columnas que NO son features
non_feature_cols = ['Occupancy', 'Date', 'Day_of_Week', 'Start_Time', 
                    'End_Time', 'Duration_min', 'Week']

# Features = todo lo demás (incluye Minutes_Since_Start, *_trend, etc.)
feature_cols = [c for c in df.columns if c not in non_feature_cols]

print(f"   Feature columns: {len(feature_cols)}")
for col in feature_cols:
    # Marcar las nuevas columnas de tiempo
    marca = ""
    if 'Minutes_Since' in col:
        marca = " 🕐"
    elif '_trend' in col:
        marca = " 📈"
    print(f"      {col}{marca}")

# ===================================================================
# 3. PREPARACIÓN DEL DATAFRAME PRINCIPAL
# ===================================================================
print("\n" + "="*60)
print("3. PREPARING FEATURE DATAFRAME")
print("="*60)

df_grouped = df[['Occupancy', 'Date'] + feature_cols].copy()
df_grouped = df_grouped.dropna(subset=feature_cols, how='all')
df_grouped = df_grouped.sort_values('Date').reset_index(drop=True)

print(f"   Samples after cleaning: {len(df_grouped)}")
print(f"   Date range: {df_grouped['Date'].min().date()} - {df_grouped['Date'].max().date()}")

# ===================================================================
# 4. SPLIT TEMPORAL 70/30
# ===================================================================
print("\n" + "="*60)
print("4. TEMPORAL SPLIT 70/30")
print("="*60)

# Ordenar por fecha
df_grouped = df_grouped.sort_values('Date').reset_index(drop=True)

# Calcular punto de corte (70% para train)
split_idx = int(len(df_grouped) * 0.7)
split_date = df_grouped.loc[split_idx, 'Date']

print(f"   Total samples: {len(df_grouped)}")
print(f"   Train (70%): {split_idx} samples")
print(f"   Test (30%): {len(df_grouped) - split_idx} samples")
print(f"   Split date: {split_date.date()}")

# Crear conjuntos
df_train = df_grouped.iloc[:split_idx].copy()
df_test  = df_grouped.iloc[split_idx:].copy()

print(f"\n   Train date range: {df_train['Date'].min().date()} - {df_train['Date'].max().date()}")
print(f"   Test date range:  {df_test['Date'].min().date()} - {df_test['Date'].max().date()}")

# Verificar que no hay solapamiento
overlap = set(df_train['Date'].unique()) & set(df_test['Date'].unique())
print(f"   Date overlap: {'⚠️ YES!' if overlap else '✅ NONE'}")

# ===================================================================
# 5. CREAR MATRICES X e y
# ===================================================================
print("\n" + "="*60)
print("5. CREATING X AND y MATRICES")
print("="*60)

# Rellenar NaN con la media de cada columna (solo en train)
for col in feature_cols:
    mean_val = df_train[col].mean()
    df_train[col] = df_train[col].fillna(mean_val)
    df_test[col] = df_test[col].fillna(mean_val)  # Usar media de train

# Crear matrices
X_train = df_train[feature_cols].values
y_train = df_train['Occupancy'].values
X_test  = df_test[feature_cols].values
y_test  = df_test['Occupancy'].values

print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   y_test:  {y_test.shape}")

# Guardar nombres de features
feature_names = feature_cols

# ===================================================================
# 6. GUARDAR DATASETS
# ===================================================================
print("\n" + "="*60)
print("6. SAVING DATASETS")
print("="*60)

# Guardar matrices como Excel
pd.DataFrame(X_train, columns=feature_names).to_excel(
    os.path.join(OUTPUT_DIR, 'X_train.xlsx'), index=False)
pd.DataFrame(X_test, columns=feature_names).to_excel(
    os.path.join(OUTPUT_DIR, 'X_test.xlsx'), index=False)
pd.DataFrame({'Occupancy': y_train}).to_excel(
    os.path.join(OUTPUT_DIR, 'y_train.xlsx'), index=False)
pd.DataFrame({'Occupancy': y_test}).to_excel(
    os.path.join(OUTPUT_DIR, 'y_test.xlsx'), index=False)

# También como regresión (mismo formato que otros pasos)
pd.DataFrame({'Occupancy': y_train}).to_excel(
    os.path.join(OUTPUT_DIR, 'y_train_regression.xlsx'), index=False)
pd.DataFrame({'Occupancy': y_test}).to_excel(
    os.path.join(OUTPUT_DIR, 'y_test_regression.xlsx'), index=False)

# Guardar nombres de features
with open(os.path.join(OUTPUT_DIR, 'feature_names.pkl'), 'wb') as f:
    pickle.dump(feature_names, f)

# Guardar info del split
split_info = {
    'split_date': split_date,
    'split_idx': split_idx,
    'n_train': len(df_train),
    'n_test': len(df_test),
    'feature_cols': feature_names,
    'n_features': len(feature_names),
    'train_dates': (df_train['Date'].min(), df_train['Date'].max()),
    'test_dates': (df_test['Date'].min(), df_test['Date'].max()),
}
with open(os.path.join(OUTPUT_DIR, 'split_info.pkl'), 'wb') as f:
    pickle.dump(split_info, f)

print(f"\n   Archivos guardados en '{OUTPUT_DIR}/':")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
    print(f"      {fname} ({size:.0f} KB)")

# ===================================================================
# 7. RESUMEN FINAL
# ===================================================================
print("\n" + "="*60)
print("7. SUMMARY")
print("="*60)

print(f"\n   Features utilizadas ({len(feature_names)}):")
for i, feat in enumerate(feature_names):
    marca = ""
    if 'Minutes_Since' in feat:
        marca = " 🕐 TIEMPO"
    elif '_trend' in feat:
        marca = " 📈 TENDENCIA"
    elif '_mean' in feat:
        marca = " 📊 MEDIA"
    elif '_std' in feat:
        marca = " 📊 STD"
    elif '_max' in feat:
        marca = " 📊 MAX"
    elif '_min' in feat:
        marca = " 📊 MIN"
    print(f"   {i+1:2d}. {feat}{marca}")

print(f"\n   Train: {X_train.shape[0]} muestras ({X_train.shape[1]} features)")
print(f"   Test:  {X_test.shape[0]} muestras ({X_test.shape[1]} features)")
print(f"   Split: 70/30 temporal (sin shuffle)")
print(f"   Fecha corte: {split_date.date()}")

print("\n" + "="*60)
print("✅ PASO 3 COMPLETADO")
print("="*60)


1. LOADING OCCUPANCY BLOCKS TABLE
   Rows: 123
   Columns: 69
   Occupancy range: 0 - 64
   Date range: 2026-02-09 - 2026-03-19

2. IDENTIFYING FEATURE COLUMNS
   Feature columns: 62
      Minutes_Since_Start 🕐
      Minutes_Since_First_Occupancy 🕐
      Corridor_CO2_Increment_max
      Corridor_CO2_Increment_mean
      Corridor_CO2_Increment_min
      Corridor_CO2_Increment_std
      Corridor_CO2_Increment_trend 📈
      Corridor_CO2_max
      Corridor_CO2_mean
      Corridor_CO2_min
      Corridor_CO2_std
      Corridor_CO2_trend 📈
      Corridor_Humidity_Increment_max
      Corridor_Humidity_Increment_mean
      Corridor_Humidity_Increment_min
      Corridor_Humidity_Increment_std
      Corridor_Humidity_Increment_trend 📈
      Corridor_Humidity_max
      Corridor_Humidity_mean
      Corridor_Humidity_min
      Corridor_Humidity_std
      Corridor_Humidity_trend 📈
      Corridor_Temp_Increment_max
      Corridor_Temp_Increment_mean
      Corridor_Temp_Increment_min
      Corridor_Te